In [1]:
import pandas as pd
import spacy


In [2]:
df = pd.read_csv("mayo_diseases.csv")

In [3]:
df.head()

,disease,main_link,Diagnosis_treatment_link,Doctors_departments_link,Overview,Symptoms,When to see a doctor,Causes,Risk factors,Complications,Prevention,Diagnosis,Treatment,Coping and support,Preparing for your appointment,Lifestyle and home remedies
0,Atrial fibrillation,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Atrial fibrillation (AFib) is an irregular and...,Symptoms ofAFibmay include:\nFeelings of a fas...,"If you have symptoms of atrial fibrillation, m...",To understand the causes of atrial fibrillatio...,Things that can increase the risk of atrial fi...,Blood clots are a dangerous complication of at...,Healthy lifestyle choices can reduce the risk ...,You may not know you have atrial fibrillation ...,The goals of atrial fibrillation treatment are...,NaN,If you have an irregular or pounding heartbeat...,Following a heart-healthy lifestyle can help p...
1,Hyperhidrosis,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hyperhidrosis (hi-pur-hi-DROE-sis) is excessiv...,The main symptom of hyperhidrosis is heavy swe...,Sometimes excessive sweating is a sign of a se...,Sweating is the body's mechanism to cool itsel...,Risk factors for hyperhidrosis include:\nHavin...,Complications of hyperhidrosis include:\nInfec...,NaN,Diagnosing hyperhidrosis may start with your h...,Treating hyperhidrosis may start with treating...,Hyperhidrosis can be the cause of discomfort a...,You may start by seeing your primary care prov...,The following suggestions may help control swe...
2,Bartholin's cyst,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,The Bartholin's (BAHR-toe-linz) glands are loc...,"If you have a small, noninfected Bartholin's c...",Call your doctor if you have a painful lump ne...,Experts believe that the cause of a Bartholin'...,NaN,A Bartholin's cyst or abscess may recur and ag...,There's no way to prevent a Bartholin's cyst. ...,"To diagnose a Bartholin's cyst, your doctor ma...",Often a Bartholin's cyst requires no treatment...,NaN,Your first appointment will likely be with eit...,NaN
3,Infant reflux,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,NaN,Infant reflux is when a baby spits up liquid o...,"Most of the time, infant reflux isn't a cause ...",See a healthcare professional if a baby:\nIsn'...,"In infants, the ring of muscle between the eso...",Infant reflux is common. But some things make ...,Infant reflux usually gets better on its own. ...,NaN,"To diagnose infant reflux, a healthcare profes...","For most babies, making some changes to feedin...",NaN,You may start by seeing your baby's primary he...,To minimize reflux:\nFeed your baby in an upri...
4,Hidradenitis suppurativa,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hidradenitis suppurativa (hi-drad-uh-NIE-tis s...,Hidradenitis suppurativa can affect one or sev...,Early diagnosis of hidradenitis suppurativa is...,Hidradenitis suppurativa develops when hair fo...,Factors that increase your chance of developin...,Persistent and severe hidradenitis suppurativa...,NaN,Hidradenitis suppurativa can be mistaken for p...,"Treatment with medicines, surgery or both can ...",Hidradenitis suppurativa can be a challenge to...,You'll likely first see your primary care prov...,Mild hidradenitis suppurativa can sometimes be...


In [4]:
symptoms = df["Symptoms"]

In [5]:
text = symptoms[0]

In [6]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("Symptoms of AFib include chest pain and dizziness.")
symptoms = [ent.text for ent in doc.ents if ent.label_ == "SYMPTOM"]

In [7]:
symptoms

[]

In [8]:
from transformers import BertTokenizer, BertForSequenceClassification
from huggingface_hub import hf_hub_download
import torch

# Load model
model = BertForSequenceClassification.from_pretrained("maiserry/DoctorBot_symptoms_disease")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Predict
text = "My skin is itchy and I have a red rash"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1)
print("Predicted class:", prediction.item())


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Predicted class: 209


In [12]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "d4data/biomedical-ner-all"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

entities = ner(text)

for e in entities:
    print(e['word'], e['entity_group'])


Device set to use cpu


skin Biological_structure
it Sign_symptom
##chy Sign_symptom
red Sign_symptom
rash Sign_symptom


In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "d4data/biomedical-ner-all"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

text = 'symptom ofafibmay include feeling fast fluttering pound heartbeat call palpitation chest pain dizziness fatigue lightheadedness reduce ability exercise shortness breath weakness people atrial fibrillation afib notice symptom atrial fibrillation occasional call paroxysmal atrial fibrillation afibsymptom come symptom usually minute hour people symptom long week episode happen repeatedly symptom away people occasionalafibneed treatment persistent irregular heartbeat constant heart rhythm reset symptom occur medical treatment need correct heart rhythm long stand persistent type ofafibis constant last long month medicine procedure need correct irregular heartbeat permanent type atrial fibrillation irregular heart rhythm reset medicine need control heart rate prevent blood clot'

entities = ner(text)
for e in entities:
    if e['entity_group'] == 'Sign_symptom':
        print(e['word'], e['entity_group'])


Device set to use cpu


palpitation Sign_symptom
pain Sign_symptom
dizziness Sign_symptom
fatigue Sign_symptom
lightheadedness Sign_symptom
exercise Sign_symptom
shortness breath weakness Sign_symptom
symptom Sign_symptom
##rial Sign_symptom
##brillation Sign_symptom
at Sign_symptom
##bs Sign_symptom
##ptom Sign_symptom
symptom Sign_symptom
symptom Sign_symptom
symptom Sign_symptom
irregular heartbeat Sign_symptom
rhythm Sign_symptom
symptom Sign_symptom
##is Sign_symptom
irregular heartbeat Sign_symptom
atrial fibrillation Sign_symptom
irregular heart rhythm Sign_symptom
clot Sign_symptom
